# Multilingual KWS — ONNX Export & ESP32 Budget

This notebook picks up the trained embedding checkpoint from `multilingual_kws_v2.ipynb` and produces the deployment artifacts.

**What it does:**
1. Loads `embedding_final.pt` from the same cache directory (Google Drive on Colab, local otherwise).
2. Exports the **embedding extractor** (backbone without the training classifier head) to float32 ONNX.
3. Applies **post-training dynamic int8 quantization** via `onnxruntime.quantization.quantize_dynamic` — weights are quantized to int8, activations stay float32. No calibration data needed.
4. Verifies numerical correctness (float32 vs int8 output cosine similarity).
5. Prints an **ESP32 deployment budget table**: model file size, estimated peak SRAM, MFLOPS.

**What it does NOT do:**
- No TensorFlow, no `onnx2tf`, no QAT, no `qnnpack`. All the dependency hell from v1 is gone.
- No full static int8 quantization (that would require a calibration dataset; dynamic int8 is sufficient for the report's deployment section).

**Prerequisites:** Run `multilingual_kws_v2.ipynb` first to produce `embedding_final.pt`.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# onnx + onnxruntime are the only new deps; torchaudio is needed for the
# feature-extraction sanity check. NO tensorflow, NO onnx2tf.
%pip install -q "onnx>=1.15" "onnxruntime>=1.17"

In [ ]:
import os, json, math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

device = torch.device('cpu')   # export always on CPU for portability
print(f'torch   : {torch.__version__}')
print(f'onnx    : {onnx.__version__}')
print(f'ort     : {ort.__version__}')

In [ ]:
# ── Colab / local path setup (mirrors multilingual_kws_v2.ipynb) ──────────────
try:
    import google.colab  # noqa: F401
    _in_colab = True
except ImportError:
    _in_colab = False

if _in_colab:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CACHE_DIR = Path('/content/drive/MyDrive/kws_cache')
    print('Colab detected — loading from Google Drive:', CACHE_DIR)
else:
    CACHE_DIR = Path('./kws_cache')
    print('Local run — loading from:', CACHE_DIR.resolve())

BUNDLE_PATH = CACHE_DIR / 'embedding_final.pt'
if not BUNDLE_PATH.exists():
    raise FileNotFoundError(
        f'{BUNDLE_PATH} not found.\n'
        'Run multilingual_kws_v2.ipynb through Section 9 first.'
    )
print(f'Checkpoint found: {BUNDLE_PATH}  ({BUNDLE_PATH.stat().st_size/1e6:.1f} MB)')

In [ ]:
# ── Reconstruct model from checkpoint ─────────────────────────────────────────
# We copy the architecture definitions here so this notebook is self-contained
# (doesn't need to import from v2 or re-run it).

class DSBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw  = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1,
                             groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.pw  = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = F.relu(self.bn1(self.dw(x)))
        x = F.relu(self.bn2(self.pw(x)))
        return x


class MultilingualEmbedding(nn.Module):
    def __init__(self, n_classes, embed_dim=256):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 48, 3, padding=1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),
            DSBlock(48, 64),
            DSBlock(64, 64),
            nn.MaxPool2d(2),
            DSBlock(64, 128, stride=2),
            DSBlock(128, 128),
            DSBlock(128, 192),
            DSBlock(192, 256),
            nn.AdaptiveAvgPool2d(1),
        )
        self.project    = nn.Linear(256, embed_dim, bias=False)
        self.embed_bn   = nn.BatchNorm1d(embed_dim)
        self.classifier = nn.Linear(embed_dim, n_classes)

    def embed(self, x):
        h = self.backbone(x).flatten(1)
        h = self.project(h)
        h = self.embed_bn(h)
        return h

    def forward(self, x):
        emb = self.embed(x)
        return self.classifier(emb), emb


state = torch.load(BUNDLE_PATH, map_location='cpu')
n_classes  = state['n_classes']
embed_dim  = state['embed_dim']

model = MultilingualEmbedding(n_classes=n_classes, embed_dim=embed_dim)
model.load_state_dict(state['state_dict'])
model.eval()

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model loaded.  Classes: {n_classes}   Embed dim: {embed_dim}   Params: {params:,}')
print(f'Best val accuracy during training: {state.get("fewshot_summary", {}).get("overall_mean_f1", "N/A")}')

---
## Section 1 — Build the embedding-extractor subgraph

We export **only** the embedding extractor (backbone + project + embed_bn), not the classifier head. At deployment:
- The ONNX model runs on-device to convert a log-mel spectrogram → 256-dim embedding.
- The per-keyword head (`Linear(256→3)`, 771 params) is so small it is stored as raw weights in flash and evaluated with a simple dot-product + softmax — no ONNX needed for that part.

This keeps the exported model as small as possible.

In [ ]:
# ── Wrapper that exports only embed() ─────────────────────────────────────────
class EmbeddingExtractor(nn.Module):
    """Thin wrapper around model.embed() for clean ONNX export."""
    def __init__(self, full_model):
        super().__init__()
        self.backbone  = full_model.backbone
        self.project   = full_model.project
        self.embed_bn  = full_model.embed_bn

    def forward(self, x):
        h = self.backbone(x).flatten(1)
        h = self.project(h)
        h = self.embed_bn(h)
        return h


extractor = EmbeddingExtractor(model).eval()

# Sanity: output shape
dummy = torch.randn(1, 1, 49, 40)
with torch.no_grad():
    emb = extractor(dummy)
print(f'Extractor output shape: {tuple(emb.shape)}  (expect [1, {embed_dim}])')

In [ ]:
# ── Export to float32 ONNX ────────────────────────────────────────────────────
ONNX_FP32_PATH = CACHE_DIR / 'embedding_extractor_fp32.onnx'

torch.onnx.export(
    extractor,
    dummy,
    str(ONNX_FP32_PATH),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['logmel'],
    output_names=['embedding'],
    dynamic_axes={
        'logmel':    {0: 'batch'},
        'embedding': {0: 'batch'},
    },
    verbose=False,
)

# Validate graph
onnx_model = onnx.load(str(ONNX_FP32_PATH))
onnx.checker.check_model(onnx_model)
print(f'float32 ONNX graph validated.')
print(f'Nodes in graph: {len(onnx_model.graph.node)}')
print(f'File size     : {ONNX_FP32_PATH.stat().st_size / 1e3:.1f} KB')

---
## Section 2 — Dynamic int8 quantization

`quantize_dynamic` quantizes the **weight tensors** of Linear and Conv layers to int8 while keeping activations as float32. This:
- Roughly **halves the model file size** (weights go from float32 to int8).
- Speeds up CPU inference (~2× on x86 VNNI; smaller gain on ARM Cortex-M without SIMD).
- Requires **no calibration data** — the scale/zero-point for each weight is computed directly from the weight range.

For full ESP32 deployment performance, static int8 quantization (both weights *and* activations quantized) would be better — ESP-DL's int8 kernels are much faster than float32 on the Xtensa LX7. We use dynamic quantization here to keep the export notebook self-contained; the report section discusses this tradeoff.

In [ ]:
# ── Dynamic int8 quantization via onnxruntime ─────────────────────────────────
ONNX_INT8_PATH = CACHE_DIR / 'embedding_extractor_int8.onnx'

quantize_dynamic(
    model_input=str(ONNX_FP32_PATH),
    model_output=str(ONNX_INT8_PATH),
    weight_type=QuantType.QInt8,
    # Quantize Conv and Matmul ops (covers all backbone + projection weights)
    nodes_to_quantize=None,   # None = quantize all eligible ops
    per_channel=True,         # per-channel is more accurate than per-tensor for Conv
    reduce_range=False,
)

print(f'int8 ONNX written: {ONNX_INT8_PATH}')
print(f'File size         : {ONNX_INT8_PATH.stat().st_size / 1e3:.1f} KB')
print(f'Reduction         : {ONNX_FP32_PATH.stat().st_size / ONNX_INT8_PATH.stat().st_size:.1f}×')

---
## Section 3 — Correctness verification

We run a batch of random inputs through both the float32 and int8 ONNX models and check:
- **Cosine similarity** of the embeddings (should be > 0.99 for dynamic int8 — weight quantization with per-channel scaling is very accurate).
- **Max absolute error** per output dimension.

This is a necessary sanity check before reporting deployment results.

In [ ]:
# ── Numerical comparison: float32 vs int8 ONNX ────────────────────────────────
rng   = np.random.RandomState(0)
batch = rng.randn(16, 1, 49, 40).astype(np.float32)

sess_fp32 = ort.InferenceSession(str(ONNX_FP32_PATH),
                                  providers=['CPUExecutionProvider'])
sess_int8 = ort.InferenceSession(str(ONNX_INT8_PATH),
                                  providers=['CPUExecutionProvider'])

emb_fp32 = sess_fp32.run(['embedding'], {'logmel': batch})[0]  # (16, 256)
emb_int8 = sess_int8.run(['embedding'], {'logmel': batch})[0]

# Per-sample cosine similarity
cos_sim = [
    float(np.dot(emb_fp32[i], emb_int8[i]) /
          (np.linalg.norm(emb_fp32[i]) * np.linalg.norm(emb_int8[i]) + 1e-9))
    for i in range(batch.shape[0])
]
max_abs_err = float(np.abs(emb_fp32 - emb_int8).max())
mean_cos    = float(np.mean(cos_sim))
min_cos     = float(np.min(cos_sim))

print(f'Cosine similarity (fp32 vs int8):')
print(f'  mean = {mean_cos:.5f}   min = {min_cos:.5f}   (threshold: > 0.99)')
print(f'Max absolute error per dimension: {max_abs_err:.4f}')

if min_cos < 0.99:
    print('WARNING: cosine similarity below 0.99 — check quantization settings.')
else:
    print('Numerical accuracy OK.')

---
## Section 4 — ESP32 deployment budget

We estimate the memory and compute budget for deploying the embedding extractor on an **ESP32-S3** (240 MHz Xtensa LX7, 512 KB SRAM internal + up to 8 MB PSRAM external, 16 MB flash).

**Key figures to compute:**
- Flash footprint: ONNX model file size
- Peak SRAM: largest intermediate activation tensor during a single forward pass
- Inference compute: total multiply-accumulate operations (MACs)

For the 5-shot keyword head (`Linear(256→3)`): 771 parameters = 3.0 KB float32 or 771 bytes int8. Effectively free.

In [ ]:
# ── ESP32 budget table ────────────────────────────────────────────────────────
# Activation memory is estimated by tracing the model's tensor shapes.
# We use the extractor's named children to reconstruct the shape flow manually
# (avoids pulling in third-party profilers).

# --- 1. Flash footprint ---
fp32_kb = ONNX_FP32_PATH.stat().st_size / 1024
int8_kb  = ONNX_INT8_PATH.stat().st_size / 1024

# --- 2. Activation memory trace (worst-case = peak buffer in the backbone) ---
# Shape flow for input (1, 1, 49, 40) through the backbone:
# Note: MaxPool2d halves both dims; stride-2 DSBlock halves both dims.
#
# Layer                        Output shape         Bytes (float32)
# ─────────────────────────────────────────────────────────────────
# Conv2d(1→48, 3x3)            (1, 48, 49, 40)      376,320
# DSBlock(48→64)               (1, 64, 49, 40)      501,760
# DSBlock(64→64)               (1, 64, 49, 40)      501,760
# MaxPool2d(2)                 (1, 64, 24, 20)      122,880
# DSBlock(64→128, stride=2)    (1, 128, 12, 10)     61,440
# DSBlock(128→128)             (1, 128, 12, 10)     61,440
# DSBlock(128→192)             (1, 192, 12, 10)     92,160
# DSBlock(192→256)             (1, 256, 12, 10)     122,880
# AdaptiveAvgPool2d(1)         (1, 256, 1, 1)       1,024
# flatten                      (1, 256)             1,024
# project Linear(256→256)      (1, 256)             1,024
# embed_bn                     (1, 256)             1,024

shapes = [
    ('Conv2d 1→48',          (48, 49, 40)),
    ('DSBlock 48→64 (#1)',   (64, 49, 40)),
    ('DSBlock 64→64 (#2)',   (64, 49, 40)),
    ('MaxPool2d(2)',          (64, 24, 20)),
    ('DSBlock 64→128 s=2',   (128, 12, 10)),
    ('DSBlock 128→128',      (128, 12, 10)),
    ('DSBlock 128→192',      (192, 12, 10)),
    ('DSBlock 192→256',      (256, 12, 10)),
    ('AdaptiveAvgPool2d(1)', (256, 1, 1)),
    ('Project + BN',         (256,)),
]

print('Activation tensor sizes (float32, single inference, batch=1):')
print(f'  {"Layer":<28s} {"Shape":<18s} {"Bytes":>10s} {"KB":>8s}')
peak_bytes = 0
peak_layer = ''
for name, shape in shapes:
    nbytes = 4 * math.prod(shape)   # float32 = 4 bytes
    kb     = nbytes / 1024
    print(f'  {name:<28s} {str(shape):<18s} {nbytes:>10,d} {kb:>8.1f}')
    if nbytes > peak_bytes:
        peak_bytes = nbytes
        peak_layer = name

print(f'\nPeak activation tensor: {peak_layer}  ({peak_bytes/1024:.1f} KB float32)')
print(f'  → with int8 activations this drops to  ~{peak_bytes/1024/4:.1f} KB')

# --- 3. MACs estimate ---
# Depthwise-separable MACs: DW + PW, ignoring bias
def conv_macs(Cin, Cout, H, W, K=3, stride=1, groups=1):
    """MACs for one conv layer (no bias)."""
    Ho, Wo = H // stride, W // stride
    return (K * K * Cin // groups * Cout * Ho * Wo)

def ds_block_macs(Cin, Cout, H, W, stride=1):
    dw = conv_macs(Cin, Cin, H, W, K=3, stride=stride, groups=Cin)
    pw = conv_macs(Cin, Cout, H // stride, W // stride, K=1)
    return dw + pw

H, W = 49, 40
macs  = conv_macs(1, 48, H, W, K=3)                 # initial conv
macs += ds_block_macs(48, 64, H, W)                 # DS1
macs += ds_block_macs(64, 64, H, W)                 # DS2
H, W = H // 2, W // 2                               # MaxPool → 24×20
macs += ds_block_macs(64, 128, H, W, stride=2)      # DS3 stride=2 → 12×10
H, W = H // 2, W // 2
macs += ds_block_macs(128, 128, H, W)               # DS4
macs += ds_block_macs(128, 192, H, W)               # DS5
macs += ds_block_macs(192, 256, H, W)               # DS6
macs += 256 * 256                                   # project Linear

print(f'\nCompute estimate:')
print(f'  Total MACs (embedding extractor): {macs/1e6:.2f} M MACs')
print(f'  Equivalent MFLOPs (2 FLOPs/MAC) : {2*macs/1e6:.2f} MFLOPS')

# --- 4. Latency estimate ---
# ESP32-S3 Xtensa LX7: ~240 MHz, ~2 FLOPS/cycle (scalar float32)
# With int8 SIMD (ESP-DL uses the M-ISA extensions): ~8 OPS/cycle effective
freq_mhz = 240
flops_per_cycle_fp32 = 2.0
flops_per_cycle_int8 = 8.0
total_flops = 2 * macs
latency_fp32_ms = (total_flops / (freq_mhz * 1e6 * flops_per_cycle_fp32)) * 1e3
latency_int8_ms = (total_flops / (freq_mhz * 1e6 * flops_per_cycle_int8)) * 1e3

print(f'\nLatency estimate (ESP32-S3 @ {freq_mhz} MHz):')
print(f'  float32 (scalar)  : ~{latency_fp32_ms:.0f} ms per inference')
print(f'  int8 (SIMD/ESP-DL): ~{latency_int8_ms:.0f} ms per inference')
print(f'  (These are rough order-of-magnitude estimates; cache/memory-latency effects not modelled.)')

# --- 5. Summary table ---
print('\n' + '=' * 65)
print('ESP32 DEPLOYMENT BUDGET SUMMARY')
print('=' * 65)
print(f'  Backbone params                  : {sum(p.numel() for p in extractor.parameters()):>10,d}')
print(f'  5-shot head params (Linear 256→3): {256*3+3:>10,d}  (771 B float32)')
print(f'  ONNX float32 size                : {fp32_kb:>10.1f} KB')
print(f'  ONNX int8 size                   : {int8_kb:>10.1f} KB  ({fp32_kb/int8_kb:.1f}× smaller)')
print(f'  ESP32-S3 flash budget            : {" 16 MB":>10s}  → {int8_kb/1024/16*100:.1f}% used by model')
print(f'  Peak activation (float32)        : {peak_bytes/1024:>10.1f} KB  (fits in PSRAM)')
print(f'  Peak activation (int8)           : {peak_bytes/1024/4:>10.1f} KB  (may fit in SRAM)')
print(f'  Compute                          : {macs/1e6:>10.2f} M MACs')
print(f'  Estimated latency int8 (S3)      : ~{latency_int8_ms:.0f} ms')
print('=' * 65)
print('\nConclusion: int8 ONNX fits in 16 MB flash with significant headroom.')
print('Peak activation memory requires PSRAM in float32 mode but fits in')
print('internal SRAM if activations are also quantized to int8.')

In [ ]:
# ── Save artefact summary ─────────────────────────────────────────────────────
export_summary = {
    'onnx_fp32_kb':      round(fp32_kb, 1),
    'onnx_int8_kb':      round(int8_kb, 1),
    'compression_ratio': round(fp32_kb / int8_kb, 2),
    'cosine_similarity_mean': round(mean_cos, 5),
    'cosine_similarity_min':  round(min_cos,  5),
    'max_abs_error':     round(max_abs_err, 4),
    'total_macs':        macs,
    'peak_activation_kb_fp32': round(peak_bytes / 1024, 1),
    'estimated_latency_int8_ms_esp32s3': round(latency_int8_ms, 1),
    'backbone_params':   sum(p.numel() for p in extractor.parameters()),
}
summary_path = CACHE_DIR / 'export_summary.json'
summary_path.write_text(json.dumps(export_summary, indent=2), encoding='utf-8')
print(f'Export summary saved: {summary_path}')

print('\nArtefacts produced:')
for p in [ONNX_FP32_PATH, ONNX_INT8_PATH, summary_path]:
    print(f'  {str(p):70s}  {p.stat().st_size/1e3:8.1f} KB')